# Testing for the 4D Integration

In [ ]:
import numpy as np
import dadi 
from matplotlib import pyplot as plt
import dadi.Polyploidy.Integration as PolyInt
from dadi.Polyploidy import wrightfisher as WF
import time
#export LD_LIBRARY_PATH=$(echo $LD_LIBRARY_PATH | sed 's|[^:]*stubs[^:]*:||g; s|:[^:]*stubs[^:]*||g')


## Tests against dadi

In [3]:
# plotting function
# useful for debugging, but it is much easier to use assert allclose to compare phis

def phi_4D_plot(phi_poly, phi_dadi, edges, axis1, axis2, xlab, ylab, plotall=False):
    # marginalize once
    phi_dadi1 = np.sum(phi_dadi, axis=axis1)
    phi_poly1 = np.sum(phi_poly, axis=axis1)
    # and then twice
    phi_dadi1 = np.sum(phi_dadi1, axis=axis2)
    phi_poly1 = np.sum(phi_poly1, axis=axis2)

    if plotall:
        plt.pcolormesh(edges, edges, phi_poly1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('PolyInt')
        plt.show()

        plt.pcolormesh(edges, edges, phi_dadi1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('dadi')
        plt.show()

        plt.pcolormesh(edges, edges, phi_poly1.T - phi_dadi1.T, cmap='viridis', shading='auto')
        plt.colorbar(label='Density')
        plt.xlabel(xlab)
        plt.ylabel(ylab)
        plt.title('PolyInt - dadi')
        plt.show()

    plt.pcolormesh(edges, edges, (phi_poly1.T - phi_dadi1.T)/phi_dadi1.T, cmap='viridis', shading='auto')
    plt.colorbar(label='Density')
    plt.xlabel(xlab)
    plt.ylabel(ylab)
    plt.title('(PolyInt - dadi)/dadi')
    plt.show()

    print(f"Largest difference between dadi and PolyInt: {np.max(np.abs(phi_poly1.T - phi_dadi1.T))}")
    

### Test against all diploids

In [4]:
xx = dadi.Numerics.default_grid(pts=21)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=-5, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
T = .1

gamma1 = -5
gamma2 = -1
gamma3 = -2
gamma4 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5

m12 = .1
m13 = 0
m14 = 1
m21 = .01
m23 = .15
m24 = 0.4
m31 = .2
m32 = .05
m34 = 0
m41 = 0.3
m42 = 0.5
m43 = 0.2

sel_dip1 = {'gamma': gamma1, 'h': h1}
sel_dip2 = {'gamma': gamma2, 'h': h2}
sel_dip3 = {'gamma': gamma3, 'h': h3}
sel_dip4 = {'gamma': gamma4, 'h': h4}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu3 = 1.2
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu4 = 1.2
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)

phi_poly = PolyInt.four_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=dipflag, ploidyflag2=dipflag, ploidyflag3=dipflag, ploidyflag4=dipflag,
                              sel_dict1=sel_dip1, sel_dict2=sel_dip2, sel_dict3=sel_dip3, sel_dict4=sel_dip4,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func,
                              m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                              m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                              theta0=1)

phi_dadi = dadi.Integration.four_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4,
                                    h1=h1, h2=h2, h3=h3, h4=h4,
                                    m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                                    m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 0, 'Pop 3', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 1, 'Pop 2', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 2, 'Pop 2', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 1, 'Pop 1', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 2, 'Pop 1', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 2, 2, 'Pop 1', 'Pop 2')

### Then, compare to all autotetraploids with rescaled parameters and no mutations

In [ ]:
xx = dadi.Numerics.default_grid(pts=31)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=-5, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

autoflag = PolyInt.PloidyType.AUTO
T = .05

gamma1 = -5
gamma2 = 1
gamma3 = 2
gamma4 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5

m12 = .1
m13 = 0
m14 = 1
m21 = .01
m23 = .15
m24 = 0.4
m31 = .2
m32 = .05
m34 = 0
m41 = 0.3
m42 = 0.5
m43 = 0.2
# these will return additive dominance
sel1 = {'gamma': gamma1}
sel2 = {'gamma': gamma2}
sel3 = {'gamma': gamma3}
sel4 = {'gamma': gamma4}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu1_func_auto = lambda t: np.exp(np.log(nu1)*t/(2*T))
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu2_func_auto = lambda t: np.exp(np.log(nu2)*t/(2*T))
nu3 = 0.7
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu3_func_auto = lambda t: np.exp(np.log(nu3)*t/(2*T))
nu4 = 0.7
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu4_func_auto = lambda t: np.exp(np.log(nu4)*t/(2*T))   

phi_poly = PolyInt.four_pops(phi.copy(), xx, T=2*T, 
                              ploidyflag1=autoflag, ploidyflag2=autoflag, ploidyflag3=autoflag, ploidyflag4=autoflag,
                              sel_dict1=sel1, sel_dict2=sel2, sel_dict3=sel3, sel_dict4=sel4,
                              nu1=nu1_func_auto, nu2=nu2_func_auto, nu3=nu3_func_auto, nu4=nu4_func_auto,
                              m12=m12/2, m21=m21/2, m13=m13/2, m31=m31/2, m23=m23/2, m32=m32/2,
                              m14=m14/2, m24=m24/2, m34=m34/2, m41=m41/2, m42=m42/2, m43=m43/2,
                              theta0=0)



phi_dadi = dadi.Integration.four_pops(phi.copy(), xx, T=T, theta0=0, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4,
                                    h1=h1, h2=h2, h3=h3, h4=h4,
                                    m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                                    m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 0, 'Pop 3', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 1, 'Pop 2', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 2, 'Pop 2', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 1, 'Pop 1', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 2, 'Pop 1', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 2, 2, 'Pop 1', 'Pop 2')

### Then, compare to a pair of allotetraploid populations

In [ ]:
xx = dadi.Numerics.default_grid(pts=31)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=-5, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

alloaflag = PolyInt.PloidyType.ALLOa
allobflag = PolyInt.PloidyType.ALLOb    
T = .05

gamma1 = 0
gamma2 = 0

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5


m13 = 0
m14 = 1
m23 = .15
m24 = 0.4
m31 = .2
m32 = .05
m41 = 0.3
m42 = 0.5
# following pairs specify a single exchange parameter
m21 = m12 = .1
m34 = m43 = 0.2
# these will return additive dominance
sel1 = {'gamma': gamma1}
sel2 = {'gamma': gamma2}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)


phi_poly = PolyInt.four_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=alloaflag, ploidyflag2=allobflag, ploidyflag3=alloaflag, ploidyflag4=allobflag,
                              sel_dict1=sel1, sel_dict2=sel1, sel_dict3=sel2, sel_dict4=sel2,
                              nu1=nu1_func, nu2=nu1_func, nu3=nu2_func, nu4=nu2_func,
                              m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                              m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                              theta0=1)

phi_dadi = dadi.Integration.four_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma1, gamma2=gamma1, gamma3=gamma2, gamma4=gamma2,
                                    h1=h1, h2=h2, h3=h3, h4=h4,
                                    m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                                    m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                                    nu1=nu1_func, nu2=nu1_func, nu3=nu2_func, nu4=nu2_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 0, 'Pop 3', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 1, 'Pop 2', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 2, 'Pop 2', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 1, 'Pop 1', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 2, 'Pop 1', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 2, 2, 'Pop 1', 'Pop 2')


### Then compare to all autohexaploids with rescaled parameters and no mutations

In [ ]:
xx = dadi.Numerics.default_grid(pts=31)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=-5, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

hexflag = PolyInt.PloidyType.AUTOHEX
T = .05

gamma1 = -5
gamma2 = 1
gamma3 = 2
gamma4 = -3

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5

m12 = .1
m13 = 0
m14 = 1
m21 = .01
m23 = .15
m24 = 0.4
m31 = .2
m32 = .05
m34 = 0
m41 = 0.3
m42 = 0.5
m43 = 0.2
# these will return additive dominance
sel1 = {'gamma': gamma1}
sel2 = {'gamma': gamma2}
sel3 = {'gamma': gamma3}
sel4 = {'gamma': gamma4}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu1_func_auto = lambda t: np.exp(np.log(nu1)*t/(3*T))
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)
nu2_func_auto = lambda t: np.exp(np.log(nu2)*t/(3*T))
nu3 = 0.7
nu3_func = lambda t: np.exp(np.log(nu3)*t/T)
nu3_func_auto = lambda t: np.exp(np.log(nu3)*t/(3*T))
nu4 = 0.7
nu4_func = lambda t: np.exp(np.log(nu4)*t/T)
nu4_func_auto = lambda t: np.exp(np.log(nu4)*t/(3*T))   

phi_poly = PolyInt.four_pops(phi.copy(), xx, T=3*T, 
                              ploidyflag1=hexflag, ploidyflag2=hexflag, ploidyflag3=hexflag, ploidyflag4=hexflag,
                              sel_dict1=sel1, sel_dict2=sel2, sel_dict3=sel3, sel_dict4=sel4,
                              nu1=nu1_func_auto, nu2=nu2_func_auto, nu3=nu3_func_auto, nu4=nu4_func_auto,
                              m12=m12/3, m21=m21/3, m13=m13/3, m31=m31/3, m23=m23/3, m32=m32/3,
                              m14=m14/3, m24=m24/3, m34=m34/3, m41=m41/3, m42=m42/3, m43=m43/3,
                              theta0=0)


phi_dadi = dadi.Integration.four_pops(phi.copy(), xx, T=T, theta0=0, 
                                    gamma1=gamma1, gamma2=gamma2, gamma3=gamma3, gamma4=gamma4,
                                    h1=h1, h2=h2, h3=h3, h4=h4,
                                    m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                                    m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu3_func, nu4=nu4_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 0, 'Pop 3', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 1, 'Pop 2', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 2, 'Pop 2', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 1, 'Pop 1', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 2, 'Pop 1', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 2, 2, 'Pop 1', 'Pop 2')

### Finally, compare a diploid and a 2+2+2 hexaploid to all diploids without selection

In [16]:
xx = dadi.Numerics.default_grid(pts=21)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

dipflag = PolyInt.PloidyType.DIPLOID
hexaflag = PolyInt.PloidyType.HEXa
hexbflag = PolyInt.PloidyType.HEXb
hexcflag = PolyInt.PloidyType.HEXc  

T = .05

gamma = 0

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5

m12 = .1
m13 = 0
m14 = 1
m21 = .1
m31 = .2
m41 = 0.3

# following pairs specify exchange parameters
m23 = m32 = .05
m24 = m42 = 0.5
m43 = m34 = 0.2
# these will return additive dominance
sel = {'gamma': gamma}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)


phi_poly = PolyInt.four_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=dipflag, ploidyflag2=hexaflag, ploidyflag3=hexbflag, ploidyflag4=hexcflag,
                              sel_dict1=sel, sel_dict2=sel, sel_dict3=sel, sel_dict4=sel,
                              nu1=nu1_func, nu2=nu2_func, nu3=nu2_func, nu4=nu2_func,
                              m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                              m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                              theta0=1)

phi_dadi = dadi.Integration.four_pops(phi.copy(), xx, T=T, theta0=1, 
                                    gamma1=gamma, gamma2=gamma, gamma3=gamma, gamma4=gamma,
                                    h1=h1, h2=h2, h3=h3, h4=h4,
                                    m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                                    m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                                    nu1=nu1_func, nu2=nu2_func, nu3=nu2_func, nu4=nu2_func)

assert np.allclose(phi_poly, phi_dadi)

# mid = 0.5 * (xx[1:] + xx[:-1])
# first = xx[0] - (mid[0] - xx[0])
# last = xx[-1] + (xx[-1] - mid[-1])
# edges = np.concatenate([[first], mid, [last]])

# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 0, 'Pop 3', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 1, 'Pop 2', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 0, 2, 'Pop 2', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 1, 'Pop 1', 'Pop 4')
# phi_4D_plot(phi_poly, phi_dadi, edges, 1, 2, 'Pop 1', 'Pop 3')
# phi_4D_plot(phi_poly, phi_dadi, edges, 2, 2, 'Pop 1', 'Pop 2')


## Tests for GPU Code (not against dadi)

In [ ]:
# Test Alloautohexaploids (4+2 hexaploids)
xx = dadi.Numerics.default_grid(pts=16)
phi = dadi.PhiManip.phi_1D(xx, theta0=1, gamma=0, h=0.5)
phi = dadi.PhiManip.phi_1D_to_2D(xx, phi)
phi = dadi.PhiManip.phi_2D_to_3D(phi, 0.5, xx, xx, xx)
phi = dadi.PhiManip.phi_3D_to_4D(phi, 1/3, 1/3, xx, xx, xx, xx)

hex_tetraflag = PolyInt.PloidyType.HEX_tetra
hex_dipflag = PolyInt.PloidyType.HEX_dip

T = .05

gamma1 = -2
gamma2 = .5

h1 = 0.5
h2 = 0.5
h3 = 0.5
h4 = 0.5

m13 = 0
m31 = 0.2
m42 = 0.3
m24 = 0.1
m14 = m41 = m23 = m32 = 0

# following pairs specify exchange parameters
m12 = m21 = .05
m43 = m34 = 0.25
# these will return additive dominance
sel1 = {'gamma': gamma1}
sel2 = {'gamma': gamma2}

nu1 = 1.5
nu1_func = lambda t: np.exp(np.log(nu1)*t/T)
nu2 = 3.0
nu2_func = lambda t: np.exp(np.log(nu2)*t/T)

phi_cpu = PolyInt.four_pops(phi.copy(), xx, T=T, 
                              ploidyflag1=hex_tetraflag, ploidyflag2=hex_dipflag, ploidyflag3=hex_tetraflag, ploidyflag4=hex_dipflag,
                              sel_dict1=sel1, sel_dict2=sel1, sel_dict3=sel2, sel_dict4=sel2,
                              nu1=nu1_func, nu2=nu1_func, nu3=nu2_func, nu4=nu2_func,
                              m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
                              m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
                              theta0=1)

# PolyInt.cuda_enabled = True
# phi_gpu = PolyInt.four_pops(phi.copy(), xx, T=T, 
#                               ploidyflag1=hex_tetraflag, ploidyflag2=hex_dipflag, ploidyflag3=hex_tetraflag, ploidyflag4=hex_dipflag,
#                               sel_dict1=sel1, sel_dict2=sel1, sel_dict3=sel2, sel_dict4=sel2,
#                               nu1=nu1_func, nu2=nu1_func, nu3=nu2_func, nu4=nu2_func,
#                               m12=m12, m21=m21, m13=m13, m31=m31, m23=m23, m32=m32,
#                               m14=m14, m24=m24, m34=m34, m41=m41, m42=m42, m43=m43,
#                               theta0=1)